# GPT-2 Fine-tuning (Causal Language Modelling on Reuters Descriptions)

GPT-2 is **decoder-only**. Attention is masked so each token sees only what came before it, and the
single training objective is to predict the next token. There is no classification head and no
separate encoder — the model learns the shape of the text itself, which is why the target for this
pipeline is the input, shifted.

That makes it the odd one out here in a way worth noticing: `train_bert.ipynb` produces a `labels`
column of integers and `train_t5.ipynb` produces one of tokenized headlines, but this pipeline
produces **no labels at all**. The collator builds them at batch time.

> **Before you run this:** switch on the GPU with **Runtime -> Change runtime type -> Hardware
> accelerator -> GPU**.

## Clone the project

Colab reads `main`, so anything you have not pushed is invisible here. The URL is **HTTPS** — the
`git@github.com:` remote on your machine needs an SSH key Colab does not have.

`data/guardian_headlines.csv` and `data/reuters_headlines.csv` are tracked, so the clone brings the
data with it and there is nothing to upload.

In [ ]:
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

## Install the package from the clone

The install has to be **editable** (`-e`). `headlines/config.py` computes

```python
PROJECT_ROOT = Path(__file__).resolve().parents[2]
```

so the CSV paths are found relative to wherever `config.py` physically lives. An editable install
leaves it inside the clone next to `data/`; a regular install copies it into `site-packages`, where
`parents[2]` points somewhere with no `data/` directory at all.

In [ ]:
%pip install -q -e /content/Bert-T5-GPT2

## Check the environment before training

Two separate things can go wrong here and they look nothing alike.

If pip upgraded a package that Colab had *already imported*, the old module object stays in memory
for the rest of the session and the new version only loads after **Runtime -> Restart session**. The
version check reads what is actually importable, not what pip claims it installed, so it catches that.

The second is precision. `resolve_precision()` in `headlines/utils.py` reads exactly the two CUDA
values below: bf16 where the device supports it, fp32 everywhere else — never fp16. ModernBERT and T5
were pretrained in bf16 and emit NaN losses under fp16, and that failure does not raise — training
runs to completion and the score comes back at zero. A T4 has no bf16, so it uses fp32 and takes
longer. GPT-2 itself is fp16-safe; if you want the speed back, build the training arguments with
`fp16=True`, which `resolve_precision` then leaves alone.

In [ ]:
import gc
from importlib.metadata import version

import torch


MINIMUMS = {"transformers": (4, 48), "datasets": (3, 0), "accelerate": (1, 0), "torch": (2, 2)}

for name, minimum in MINIMUMS.items():
    installed = version(name)
    parts = tuple(int(piece) for piece in installed.split(".")[:2])
    status = "ok" if parts >= minimum else "TOO OLD - restart the session"
    print(f"{name:<14} {installed:<12} (need >= {'.'.join(map(str, minimum))})  {status}")

print()
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device:        {torch.cuda.get_device_name(0)}")
    print(f"bf16 support:  {torch.cuda.is_bf16_supported()}")

## Configure the run

`accum_steps=4` is the field to look at. With `batch_size=12` the optimizer only steps after four
batches have accumulated gradients, so the model trains as if the batch were 48 while never holding
more than 12 sequences of 256 tokens in memory at once. It costs wall-clock, not VRAM.

In [ ]:
from headlines.config import training_arguments
from headlines.gpt2.config import CLM, ClmDataArguments, ClmModelArguments


model_args = ClmModelArguments()
data_args = ClmDataArguments()
training_args = training_arguments(CLM, output_dir="/content/artifacts/gpt2")

batch = training_args.per_device_train_batch_size
accum = training_args.gradient_accumulation_steps
print(f"effective batch size: {batch} x {accum} = {batch * accum}")
print(model_args)
print(data_args)

## Load the descriptions

Only the `Description` column matters here. There is no input/output pair to build — the model is
being fitted to the distribution of this text, so the text is the whole dataset.

In [ ]:
from headlines.data import load_csv


frame = load_csv(data_args.data_path, [data_args.text_column])
print(f"{len(frame):,} descriptions after cleaning")
print()
print(frame.loc[0, data_args.text_column])

## The tokenizer needs a pad token

GPT-2 ships without one, because it was trained on continuous streams of text that never needed
padding. Batching fails outright until `pad_token` is set, so `load_tokenizer` reuses the
end-of-text token. That keeps the vocabulary the same size, which means no `resize_token_embeddings`
and no randomly initialised row in the embedding matrix.

In [ ]:
from headlines.gpt2.dataset import build_datasets
from headlines.gpt2.utils.tokenizer import load_tokenizer


tokenizer = load_tokenizer(model_args)
print(f"pad_token = {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})")
print(f"eos_token = {tokenizer.eos_token!r}  (id {tokenizer.eos_token_id})")

datasets = build_datasets(data_args, tokenizer, training_args.seed)
for name, split in datasets.items():
    print(f"{name:<11} {len(split):>7,} rows  columns={split.column_names}")

### Where the labels come from

No `labels` column — unlike the other two pipelines. `DataCollatorForLanguageModeling`
with `mlm=False` fills that in when a batch is assembled: it copies `input_ids` into `labels`, then
overwrites every padded position with `-100`, the value PyTorch's cross-entropy ignores. The model
handles the shift-by-one internally, so nothing here has to offset anything.

Run the collator on two rows and look at what comes back.

In [ ]:
from transformers import DataCollatorForLanguageModeling


collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
batch = collator([datasets["train"][index] for index in range(2)])

print("keys:", list(batch.keys()))
for name, tensor in batch.items():
    print(f"{name:<16} {tuple(tensor.shape)}")

shortest = batch["attention_mask"][0].sum().item()
print()
print("input_ids tail:", batch["input_ids"][0][shortest - 3 :].tolist())
print("labels    tail:", batch["labels"][0][shortest - 3 :].tolist())
print()
print(f"padded positions masked to -100: {(batch['labels'] == -100).sum().item():,}")

## Build the Trainer

There is no `compute_metrics` for this pipeline. The `Trainer` already reports `eval_loss`, and the
metric anyone actually quotes for a language model — perplexity — is just `exp(loss)`, so
`headlines/gpt2/metrics.py` exposes that one function instead of a metrics callback.

Because lower is better here, `build_trainer` passes `greater_is_better=False` alongside
`metric_for_best_model="eval_loss"`. Leaving that out would make early stopping keep the *worst*
checkpoint.

In [ ]:
from headlines.gpt2.train import build_trainer


trainer = build_trainer(model_args, data_args, training_args)
print(f"training on {len(trainer.train_dataset):,} rows for {training_args.num_train_epochs:g} epochs")
print(f"precision: fp16={trainer.args.fp16} bf16={trainer.args.bf16}")

## Smoke test before the real run

The full run is 2,172 batches per epoch over two epochs — roughly 15 to 25 minutes on a T4. Finding
out at minute 12 that a collator or a metric argument is wrong is a bad way to spend an afternoon.

This builds a second `Trainer` that runs twenty steps and stops. The overrides are ordinary
`TrainingArguments` fields set on a second arguments object, so anything the library accepts can be
passed here and the run above is left untouched.

The strategies have to move together. `CLM` sets `load_best_model_at_end=True`,
which requires `eval_strategy` and `save_strategy` to match — and leaving both on `"epoch"` while
capping at twenty steps means no epoch ever completes, so there is no checkpoint to load at the end.
Switching both to `"steps"` gives the run something to select from.

The trainer is deleted at the end because it is a second full GPT-2 with its own AdamW state — a couple of gigabytes that would otherwise sit on the card for the whole real run. `gc.collect()` has to come first; `empty_cache` only releases blocks nothing still references.

If this cell finishes and prints a loss, the pipeline is sound and the long run is worth starting.

In [ ]:
smoke_args = training_arguments(
    CLM,
    output_dir="/content/smoke",
    max_steps=20,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=10,
    save_steps=10,
    logging_steps=5,
)
smoke = build_trainer(model_args, data_args, smoke_args)
smoke.train()

del smoke
gc.collect()
torch.cuda.empty_cache()

## Measure the baseline before training

Fine-tuning a language model produces no accuracy figure to point at, so a single perplexity number
afterwards is uninterpretable — there is nothing to compare it against. Running `evaluate()` before
`train()` records what off-the-shelf GPT-2 scores on this text, and the drop from that to the tuned
number is the only honest evidence the fine-tune achieved anything.

`trainer.test_dataset` is the held-out split `build_trainer` sets aside. Early stopping reads
`validation` and never this, so the same rows can be scored before and after the run.

In [ ]:
from headlines.gpt2.metrics import perplexity


baseline = trainer.evaluate(trainer.test_dataset, metric_key_prefix="baseline")
baseline_ppl = perplexity(baseline["baseline_loss"])

print(f"pretrained GPT-2 on held-out Reuters text: loss {baseline['baseline_loss']:.4f}  perplexity {baseline_ppl:.2f}")

## Fine-tune

In [ ]:
trainer.train()

## Loss and perplexity

These are the same curve on two scales — perplexity is `exp` of the loss. It is plotted anyway
because the units mean something: a perplexity of 30 says the model is about as uncertain as if it
were choosing uniformly among 30 tokens at each step.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


history = pd.DataFrame(trainer.state.log_history)
steps = history.dropna(subset=["loss"])
evals = history.dropna(subset=["eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps["epoch"], steps["loss"], label="train", alpha=0.6)
axes[0].plot(evals["epoch"], evals["eval_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

scores = [perplexity(loss) for loss in evals["eval_loss"]]
axes[1].plot(evals["epoch"], scores, marker="o", color="darkorange")
axes[1].axhline(baseline_ppl, color="grey", linestyle="--", label="pretrained baseline (test)")
axes[1].set_title("Validation perplexity")
axes[1].legend()
axes[1].set_xlabel("epoch")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.show()

## Test perplexity after fine-tuning

The same split the baseline was measured on, so the two numbers are directly comparable.
`load_best_model_at_end` is on, so this scores the best checkpoint rather than whatever the last
epoch happened to produce. These rows were never trained on and never selected against — early
stopping read `validation`, not this — so the reduction below is the improvement, with no thumb on
the scale.

In [ ]:
tuned = trainer.evaluate(trainer.test_dataset, metric_key_prefix="test")
tuned_ppl = perplexity(tuned["test_loss"])

print(f"pretrained  loss {baseline['baseline_loss']:.4f}  perplexity {baseline_ppl:.2f}")
print(f"fine-tuned  loss {tuned['test_loss']:.4f}  perplexity {tuned_ppl:.2f}")
print(f"reduction   {100 * (1 - tuned_ppl / baseline_ppl):.1f}%")

### Generate continuations

Perplexity can look healthy while the model loops on a phrase or drifts into boilerplate, because
repeating a confident token is cheap under the loss. Sampling with `top_p` and forbidding repeated
trigrams pushes back on that, but the only real check is reading the output.

Two things this cell has to get right.

**Padding switches to the left.** GPT-2 pads on the right by default, which is correct for training
and wrong for batched generation: continuation starts from the last position, so a right-padded
prompt has the model continuing from padding rather than from the final real token. Short prompts in
a batch come out as nonsense while the longest one looks fine, which makes it an easy bug to miss.

**`pad_token_id` is passed explicitly.** It falls back to the model config's value, which GPT-2
leaves unset, so generation warns on every call and can run past where it should stop.

In [ ]:
model = trainer.model
torch.manual_seed(training_args.seed)

prompts = [
    "The central bank said on Thursday",
    "Shares of the technology group fell",
    "The trade agreement between",
]

tokenizer.padding_side = "left"
encoded = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

generated_ids = model.generate(
    **encoded,
    max_new_tokens=60,
    do_sample=True,
    top_p=0.92,
    temperature=0.8,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.pad_token_id,
)

for text in tokenizer.batch_decode(generated_ids, skip_special_tokens=True):
    print(text)
    print()

## Save and download the checkpoint

`artifacts/` is gitignored, and a Colab runtime takes its disk with it when it disconnects. Download
the archive, or mount Drive and copy it there, before you close the tab.

In [ ]:
import shutil

from google.colab import files


trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

archive = shutil.make_archive("/content/gpt2-headlines", "zip", training_args.output_dir)
files.download(archive)